# Team B Solution — Translation Symmetry, Bloch Momentum, and the Order-Finding Core Behind Shor

We solve the workshop in four explicit stages:

1. build the translation operator and tight-binding Hamiltonian,
2. verify the Bloch-momentum picture classically,
3. use QFT as the position-to-momentum basis change,
4. use QPE to recover the translation eigenphase and connect it to Shor’s order-finding logic.

The key operator identities are

$$
T|j\rangle = |j+1 \bmod 8\rangle,
\qquad
T^8 = I,
$$

and the Bloch states satisfy

$$
T|k_n\rangle = e^{-i2\pi n/8}|k_n\rangle.
$$

In [ ]:
import os
import warnings
from fractions import Fraction
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mpl-cache"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)
os.environ.setdefault("XDG_CACHE_HOME", str(Path.cwd() / ".cache"))
Path(os.environ["XDG_CACHE_HOME"]).mkdir(exist_ok=True)
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import QFTGate
from qiskit.circuit.library.generalized_gates import UnitaryGate

warnings.filterwarnings("ignore", message="FigureCanvasAgg is non-interactive, and thus cannot be shown")

np.set_printoptions(precision=5, suppress=True)
pd.options.display.float_format = "{:.6f}".format
plt.style.use("tableau-colorblind10")

## Step 1 — Build the Translation Operator and the Tight-Binding Hamiltonian

The physical system is an 8-site periodic ring with nearest-neighbor hopping amplitude $J=1$.

We first write the matrices $T$ and $H$ explicitly in the site basis.

In [ ]:
N = 8
J = 1.0

T = np.zeros((N, N), dtype=complex)
for j in range(N):
    T[(j + 1) % N, j] = 1.0

H = np.zeros((N, N), dtype=complex)
for j in range(N):
    H[(j + 1) % N, j] -= J
    H[j, (j + 1) % N] -= J

print("Translation operator T:")
print(T)
print("\nTight-binding Hamiltonian H:")
print(H)

## Step 2 — Construct the Bloch States and the Analytic Energies

For each momentum label $n=0,\dots,7$, the Bloch state is

$$
|k_n\rangle = \frac{1}{\sqrt{8}}\sum_{j=0}^{7} e^{i2\pi nj/8}|j\rangle.
$$

The corresponding tight-binding energy is

$$
E_n = -2\cos\left(\frac{2\pi n}{8}\right).
$$

In [ ]:
bloch_states = []
translation_eigenvalues = []
analytic_energies = []

for n in range(N):
    sites = np.arange(N)
    bloch_vector = np.exp(1j * 2 * np.pi * n * sites / N) / np.sqrt(N)
    bloch_states.append(bloch_vector)
    translation_eigenvalues.append(np.exp(-1j * 2 * np.pi * n / N))
    analytic_energies.append(-2 * J * np.cos(2 * np.pi * n / N))

bloch_states = np.column_stack(bloch_states)
translation_eigenvalues = np.array(translation_eigenvalues)
analytic_energies = np.array(analytic_energies)

translation_table = pd.DataFrame(
    {
        "n": np.arange(N),
        "Eigenvalue of T": translation_eigenvalues,
        "Analytic energy": analytic_energies,
    }
)
display(translation_table)

## Step 3 — Verify the Symmetry Relations Numerically

The classical benchmark checks are:

- $T^8 = I$,
- $[H,T] = 0$,
- the Bloch states line up with the exact Hamiltonian eigenvectors.

In [ ]:
ring_checks = pd.DataFrame(
    [
        ["||T^8 - I||", np.linalg.norm(np.linalg.matrix_power(T, 8) - np.eye(N))],
        ["||[H,T]||", np.linalg.norm(H @ T - T @ H)],
    ],
    columns=["Check", "Value"],
)
display(ring_checks)

H_evals, H_evecs = np.linalg.eigh(H)

overlap_rows = []
for n in range(N):
    overlaps = np.abs(H_evecs.conj().T @ bloch_states[:, n]) ** 2
    overlap_rows.append([n, analytic_energies[n], overlaps.max()])

overlap_table = pd.DataFrame(
    overlap_rows,
    columns=["Momentum label n", "Analytic energy", "Best overlap with exact eigenbasis"],
)
display(overlap_table)

## Step 4 — Visualize the Translation Spectrum

The left panel places the translation eigenvalues on the complex unit circle.

The right panel shows the tight-binding dispersion relation $E_n$ across the 8 allowed momentum labels.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(translation_eigenvalues.real, translation_eigenvalues.imag, s=90, color="#d97706")
circle = plt.Circle((0, 0), 1.0, color="#cbd5e1", fill=False, linewidth=1.5)
axes[0].add_patch(circle)
for n, eig in enumerate(translation_eigenvalues):
    axes[0].text(eig.real + 0.04, eig.imag + 0.04, f"n={n}", fontsize=10)
axes[0].axhline(0, color="#94a3b8", linewidth=1)
axes[0].axvline(0, color="#94a3b8", linewidth=1)
axes[0].set_aspect("equal")
axes[0].set_xlim(-1.25, 1.25)
axes[0].set_ylim(-1.25, 1.25)
axes[0].set_title("Translation eigenvalues on the unit circle", loc="left")
axes[0].set_xlabel("Real part")
axes[0].set_ylabel("Imaginary part")

axes[1].plot(np.arange(N), analytic_energies, marker="o", linewidth=2, color="#2563eb")
axes[1].set_xticks(np.arange(N))
axes[1].set_xlabel("Momentum label n")
axes[1].set_ylabel("Energy")
axes[1].set_title("Tight-binding dispersion", loc="left")
axes[1].grid(alpha=0.25)

fig.tight_layout()
display(fig)
plt.close(fig)

## Step 5 — Use the QFT as the Momentum Transform

A site-localized state does not have a sharp momentum label.

The QFT converts site-basis information into momentum-basis amplitudes and phases, so we study:

- $|3\rangle$,
- the superposition $(|0\rangle + |1\rangle)/\sqrt{2}$.

In [ ]:
qft_gate = QFTGate(3)

site_state_3 = np.zeros(N, dtype=complex)
site_state_3[3] = 1.0
qft_of_site_3 = Statevector(site_state_3).evolve(qft_gate)

qft_site_table = pd.DataFrame(
    {
        "Momentum index": np.arange(N),
        "Amplitude": qft_of_site_3.data,
        "Probability": np.abs(qft_of_site_3.data) ** 2,
        "Phase / pi": np.angle(qft_of_site_3.data) / np.pi,
    }
)
display(qft_site_table)

superposition_state = (np.eye(N, dtype=complex)[0] + np.eye(N, dtype=complex)[1]) / np.sqrt(2)
qft_of_superposition = Statevector(superposition_state).evolve(qft_gate)

qft_superposition_table = pd.DataFrame(
    {
        "Momentum index": np.arange(N),
        "Probability": np.abs(qft_of_superposition.data) ** 2,
        "Phase / pi": np.angle(qft_of_superposition.data) / np.pi,
    }
)
display(qft_superposition_table)

## Step 6 — Plot the QFT Output Distributions

We want to see both probabilities and phases.

For a site state, the probability spread is flat, while the phase pattern carries the positional information.
For a superposition state, interference changes the probability distribution itself.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))

axes[0, 0].bar(np.arange(N), np.abs(qft_of_site_3.data) ** 2, color="#d97706")
axes[0, 0].set_title("QFT(|3>) probabilities", loc="left")
axes[0, 0].set_ylabel("Probability")
axes[0, 0].set_xticks(np.arange(N))
axes[0, 0].grid(axis="y", alpha=0.25)

axes[0, 1].bar(np.arange(N), np.angle(qft_of_site_3.data) / np.pi, color="#92400e")
axes[0, 1].set_title("QFT(|3>) phases / π", loc="left")
axes[0, 1].set_ylabel("Phase / π")
axes[0, 1].set_xticks(np.arange(N))
axes[0, 1].grid(axis="y", alpha=0.25)

axes[1, 0].bar(np.arange(N), np.abs(qft_of_superposition.data) ** 2, color="#2563eb")
axes[1, 0].set_title(r"QFT((|0\rangle + |1\rangle)/\sqrt{2}) probabilities", loc="left")
axes[1, 0].set_xlabel("Momentum index")
axes[1, 0].set_ylabel("Probability")
axes[1, 0].set_xticks(np.arange(N))
axes[1, 0].grid(axis="y", alpha=0.25)

axes[1, 1].bar(np.arange(N), np.angle(qft_of_superposition.data) / np.pi, color="#1d4ed8")
axes[1, 1].set_title(r"QFT((|0\rangle + |1\rangle)/\sqrt{2}) phases / π", loc="left")
axes[1, 1].set_xlabel("Momentum index")
axes[1, 1].set_ylabel("Phase / π")
axes[1, 1].set_xticks(np.arange(N))
axes[1, 1].grid(axis="y", alpha=0.25)

fig.tight_layout()
display(fig)
plt.close(fig)

## Step 7 — Build the QPE Circuit for a Known Bloch State

We choose the Bloch sector $n=3$.

Since

$$
T|k_3\rangle = e^{-i2\pi\cdot 3/8}|k_3\rangle,
$$

QPE should return

$$
\phi = \operatorname{frac}\!\left(-\frac{3}{8}\right) = \frac{5}{8}.
$$

In [ ]:
phase_qubits = 6
target_n = 3
target_bloch_state = bloch_states[:, target_n]

qpe_translation = QuantumCircuit(phase_qubits + 3)
for q in range(phase_qubits):
    qpe_translation.h(q)

qpe_translation.initialize(target_bloch_state, [phase_qubits, phase_qubits + 1, phase_qubits + 2])

for q in range(phase_qubits):
    powered_gate = UnitaryGate(np.linalg.matrix_power(T, 2**q), label=f"T^{2**q}")
    qpe_translation.append(powered_gate.control(1), [q, phase_qubits, phase_qubits + 1, phase_qubits + 2])

qpe_translation.append(QFTGate(phase_qubits).inverse(), range(phase_qubits))

print(qpe_translation.draw(output="text"))

## Step 8 — Run QPE and Decode the Phase

We now extract the phase-register distribution and translate the dominant phase back into the momentum label.

In [ ]:
translation_distribution = Statevector.from_instruction(qpe_translation).probabilities_dict(qargs=list(range(phase_qubits)))

translation_top = sorted(
    [(str(bitstring), float(probability)) for bitstring, probability in translation_distribution.items()],
    key=lambda item: item[1],
    reverse=True,
)[:8]

translation_rows = []
for bitstring, probability in translation_top:
    phase = int(bitstring, 2) / (2**phase_qubits)
    inferred_n = (-round(phase * N)) % N
    exact_fraction = Fraction(int(bitstring, 2), 2**phase_qubits).limit_denominator(8)
    translation_rows.append([bitstring, phase, probability, inferred_n, str(exact_fraction)])

translation_qpe_table = pd.DataFrame(
    translation_rows,
    columns=["Bitstring", "Phase", "Probability", "Recovered n", "Phase as fraction"],
)
display(translation_qpe_table)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(translation_qpe_table["Bitstring"], translation_qpe_table["Probability"], color="#0f766e")
ax.set_title("QPE histogram for the Bloch state with n = 3", loc="left")
ax.set_xlabel("Measured phase bitstring")
ax.set_ylabel("Probability")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
display(fig)
plt.close(fig)

dominant_row = translation_qpe_table.iloc[0]
display(Markdown(
    rf'''
    Dominant result:

    - bitstring: `{dominant_row['Bitstring']}`
    - phase: `{dominant_row['Phase']:.6f} = {dominant_row['Phase as fraction']}`
    - recovered momentum label: `{int(dominant_row['Recovered n'])}`

    This matches the prepared Bloch state with $n=3$.
    '''
))

## Step 9 — Use a Superposition of Momentum Sectors

Finally, we prepare a superposition of two Bloch states.

The purpose is to show that QPE resolves the finite-order phase structure into multiple peaks, exactly as one expects in order-finding problems.

In [ ]:
mixed_state = (bloch_states[:, 2] + bloch_states[:, 5]) / np.sqrt(2)

mixed_qpe = QuantumCircuit(phase_qubits + 3)
for q in range(phase_qubits):
    mixed_qpe.h(q)

mixed_qpe.initialize(mixed_state, [phase_qubits, phase_qubits + 1, phase_qubits + 2])

for q in range(phase_qubits):
    powered_gate = UnitaryGate(np.linalg.matrix_power(T, 2**q), label=f"T^{2**q}")
    mixed_qpe.append(powered_gate.control(1), [q, phase_qubits, phase_qubits + 1, phase_qubits + 2])

mixed_qpe.append(QFTGate(phase_qubits).inverse(), range(phase_qubits))

mixed_distribution = Statevector.from_instruction(mixed_qpe).probabilities_dict(qargs=list(range(phase_qubits)))

mixed_top = sorted(
    [(str(bitstring), float(probability)) for bitstring, probability in mixed_distribution.items()],
    key=lambda item: item[1],
    reverse=True,
)[:10]

mixed_rows = []
for bitstring, probability in mixed_top:
    phase = int(bitstring, 2) / (2**phase_qubits)
    inferred_n = (-round(phase * N)) % N
    mixed_rows.append([bitstring, phase, probability, inferred_n])

mixed_table = pd.DataFrame(
    mixed_rows,
    columns=["Bitstring", "Phase", "Probability", "Recovered n"],
)
display(mixed_table)

## Step 10 — Plot the Superposition Histogram and State the Shor Analogy

The superposition histogram is the visual cue that this is a finite-order phase problem.

The structural comparison with Shor is:

- a finite-order unitary,
- rational eigenphases,
- QPE to recover the phase,
- classical inference of the hidden order.

In [ ]:
comparison_table = pd.DataFrame(
    [
        ["Unitary", "Spatial translation T", "Modular multiplication U_a"],
        ["Finite order", "T^8 = I", "U_a^r = I on the cyclic subspace"],
        ["Eigenphases", "frac(-n/8) in [0,1)", "s/r in [0,1)"],
        ["Quantum primitive", "QPE on T", "QPE on modular multiplication"],
        ["Recovered structure", "Momentum / order 8", "Order r"],
    ],
    columns=["Feature", "Translation ring", "Shor core"],
)
display(comparison_table)

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(mixed_table["Bitstring"], mixed_table["Probability"], color="#7c3aed")
ax.set_title("QPE histogram for a superposition of two momentum sectors", loc="left")
ax.set_xlabel("Measured phase bitstring")
ax.set_ylabel("Probability")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
display(fig)
plt.close(fig)

## Step 11 — Final Interpretation

We have now completed the workshop step by step:

1. the translation symmetry was written explicitly,
2. the Bloch basis was verified numerically,
3. the QFT was shown as a genuine momentum transform,
4. QPE recovered the translation phase,
5. the superposition case exposed the same multi-peak logic that underlies Shor’s order-finding core.

The conceptual conclusion is precise:

> Shor’s core is not special because it is about arithmetic.  
> It is special because it is a phase-estimation method for a finite-order unitary.

Translation symmetry on a periodic ring is one of the cleanest physics examples of that structure.